In [ ]:
import openslide
import matplotlib.pyplot as plt
import numpy as np
import glob
import json
import pandas as pd
from tqdm import tqdm
from typing import List
import yaml
import os
from os.path import join as opj
import re
from matplotlib_venn import venn2, venn3
from datetime import datetime, timedelta
import warnings
import matplotlib

In [ ]:
def sanitize_string(string):
    return re.sub(r'[^a-zA-Z0-9]', '', string)

In [ ]:
def read_cat_csvs_with_fn(fns:List[str]) -> pd.DataFrame:
    return pd.concat([pd.read_csv(mfn, dtype=str, comment="#") for mfn in fns]).reset_index(drop=True)

def read_cat_csvs_with_fn_keep_fn(fns:List[str]) -> pd.DataFrame:
    all_dfs = []
    for mfn in fns:
        curr_df = pd.read_csv(mfn, dtype=str, comment="#")
        curr_df["qr_meta_fn"] = mfn.split("/")[-1]
        all_dfs.append(curr_df)
    return pd.concat(all_dfs).reset_index(drop=True)

In [ ]:
def read_cat_csvs(fn_pattern:str) -> pd.DataFrame:
    meta_fns = glob.glob(fn_pattern)
    return pd.concat([pd.read_csv(mfn, dtype=str, comment="#") for mfn in meta_fns]).reset_index(drop=True)

In [ ]:
def mxa_code_cast(x):
    if x == "": return ""
    return f"{int(x):02d}"

In [ ]:
def process_metadata(cf):
    # Initial Mx code matching
    metas = read_cat_csvs_with_fn(cf["mx_match"]["labportal"])
    metas = metas[~metas["Outside Accession"].isna()]
    metas["m_num"] = metas["Outside Accession"].apply(
        lambda x: int(x.lower().split("x")[0].removeprefix("m")))
    metas["x_num"] = metas["Outside Accession"].apply(
        lambda x: int(x.lower().split("x")[1]))

    ocr_metas = read_cat_csvs_with_fn_keep_fn(cf["mx_match"]["ocr"])
    ocr_metas_mxonly = ocr_metas[["m_code", "x_code"]]
    assert not ocr_metas_mxonly.duplicated().any()

    ocr_metas = ocr_metas.rename(columns={
        "Unnamed: 4": "tile",
        "Unnamed: 5": "comment"
    })
    ocr_metas = ocr_metas.drop("mx_code", axis=1)
    ocr_metas["m_num"] = ocr_metas["m_code"].apply(
        lambda x: int(x.lower().removeprefix("m")))
    ocr_metas["x_num"] = ocr_metas["x_code"].apply(
        lambda x: int(x.lower().removeprefix("x")))

    first_batch_all_meta = pd.merge(left=ocr_metas,
                                    right=metas,
                                    left_on=["m_num", "x_num"],
                                    right_on=["m_num", "x_num"],
                                    how="outer")

    first_batch_all_meta["a_num"] = -1
    first_batch_all_meta["mxa_code"] = first_batch_all_meta.fillna("").apply(
        lambda x: f'M{x["m_num"]}x{x["x_num"]:02d}', axis=1)
    first_batch_all_meta = first_batch_all_meta.drop(
        ["m_code", "x_code"],
        axis=1)[cf["infra"]["meta_col_order"]].sort_values(["m_num", "x_num"])

    # Mx-A code matching
    metas = read_cat_csvs_with_fn(cf["mxa_match"]["labportal"])
    metas["a_num"] = metas["A_number"].fillna("a-1").apply(
        lambda x: int(x.lower().replace("missing", "a-2").removeprefix("a")))
    metas["m_num"] = metas["Outside Accession"].apply(
        lambda x: int(x.lower().split("x")[0].removeprefix("m")))
    metas["x_num"] = metas["Outside Accession"].apply(
        lambda x: int(x.lower().split("x")[1]))

    metas_amatch = metas[~(metas["a_num"] == -1)]
    metas_mxmatch = metas[metas["a_num"] == -1].drop("a_num", axis=1)

    qrd_csv = read_cat_csvs_with_fn_keep_fn(cf["mxa_match"]["ocr"])
    qrd_csv["a_num"] = qrd_csv["a_code"].apply(
        lambda x: int(x.lower().replace("x", "a-1").removeprefix("a")))
    qrd_csv_mxmatch = qrd_csv[qrd_csv["a_num"] == -1]
    qrd_csv_amatch = qrd_csv[~(qrd_csv["a_num"] == -1)]
    qrd_csv_mxmatch["m_num"] = qrd_csv_mxmatch["m_code"].apply(
        lambda x: int(x.lower().removeprefix("m")))
    qrd_csv_mxmatch["x_num"] = qrd_csv_mxmatch["x_code"].apply(
        lambda x: int(x.lower().removeprefix("x")))

    mxa_batch_mxmatch_allmeta = pd.merge(left=qrd_csv_mxmatch,
                                         right=metas_mxmatch,
                                         left_on=["m_num", "x_num"],
                                         right_on=["m_num", "x_num"],
                                         how="outer")
    mxa_batch_mxmatch_allmeta["mxa_code"] = mxa_batch_mxmatch_allmeta.fillna(
        "").apply(lambda x: f'M{x["m_num"]}x{x["x_num"]:02d}', axis=1)
    mxa_batch_mxmatch_allmeta["tile"] = ""
    mxa_batch_mxmatch_allmeta["comment"] = ""
    mxa_batch_mxmatch_allmeta = mxa_batch_mxmatch_allmeta.drop(
        ["a_code", "m_code", "x_code", "A_number"],
        axis=1)[cf["infra"]["meta_col_order"]].sort_values(["m_num", "x_num"])

    mxa_batch_mamatch_allmeta = pd.merge(left=qrd_csv_amatch,
                                         right=metas_amatch,
                                         left_on=["a_num"],
                                         right_on=["a_num"],
                                         how="outer")
    mxa_batch_mamatch_allmeta["mxa_code"] = mxa_batch_mamatch_allmeta.fillna(
        ""
    ).apply(
        lambda x:
        f'M{mxa_code_cast(x["m_num"])}x{mxa_code_cast(x["x_num"])}A{x["a_num"]}',
        axis=1)
    mxa_batch_mamatch_allmeta["tile"] = ""
    mxa_batch_mamatch_allmeta["comment"] = ""
    mxa_batch_mamatch_allmeta["x_num"] = mxa_batch_mamatch_allmeta[
        "x_num"].fillna(-2).astype(int)
    mxa_batch_mamatch_allmeta["m_num"] = mxa_batch_mamatch_allmeta[
        "m_num"].fillna(-2).astype(int)
    mxa_batch_mamatch_allmeta["a_num"] = mxa_batch_mamatch_allmeta[
        "a_num"].fillna(-2).astype(int)
    mxa_batch_mamatch_allmeta = mxa_batch_mamatch_allmeta.drop(
        ["a_code", "m_code", "x_code", "A_number"],
        axis=1)[cf["infra"]["meta_col_order"]].sort_values(
            ["m_num", "x_num", "a_num"])

    # New normal A code matching
    metas_nn = read_cat_csvs_with_fn(cf["a_match"]["labportal"])

    metas_nn["a_num"] = metas_nn["Outside Accession"].apply(
        lambda x: int(x.lower().removeprefix("a")))
    metas_nn["m_num"] = -2
    metas_nn["x_num"] = -2

    qrd_nn = read_cat_csvs_with_fn_keep_fn(cf["a_match"]["ocr"])
    qrd_nn["a_num"] = qrd_nn["a_code"].apply(
        lambda x: int(x.lower().replace("x", "a-1").removeprefix("a")))

    mxa_newnormal_allmeta = pd.merge(left=qrd_nn,
                                     right=metas_nn,
                                     left_on=["a_num"],
                                     right_on=["a_num"],
                                     how="outer")
    mxa_newnormal_allmeta["mxa_code"] = mxa_newnormal_allmeta.fillna("").apply(
        lambda x: f'MxA{x["a_num"]}', axis=1)
    mxa_newnormal_allmeta["tile"] = ""
    mxa_newnormal_allmeta["comment"] = ""
    mxa_newnormal_allmeta["x_num"] = -1
    mxa_newnormal_allmeta["m_num"] = -1
    mxa_newnormal_allmeta["a_num"] = mxa_newnormal_allmeta["a_num"].fillna(
        -1).astype(int)
    mxa_newnormal_allmeta = mxa_newnormal_allmeta.drop(
        ["a_code", "m_code", "x_code", "A_number"],
        axis=1)[cf["infra"]["meta_col_order"]].sort_values(
            ["m_num", "x_num", "a_num"])

    all_meta = pd.concat([
        first_batch_all_meta, mxa_batch_mxmatch_allmeta,
        mxa_batch_mamatch_allmeta, mxa_newnormal_allmeta
    ]).sort_values(["a_num", "m_num", "x_num"])
    
    def proc_path_str(x):
        if type(x) == str:
            return x.removeprefix(
                "/nfs/turbo/umms-tocho/data/he.mlins_scan.svs.raw/"
            ).removeprefix("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/"
            ).removeprefix("/nfs/umms-tocho-mr/dropbox/Slide_Incoming/"
            ).replace("20241108","2024-11-08"
            ).replace("20241109","2024-11-08"
            ).replace("20241110","2024-11-08")
        else:
            return x
    all_meta["path"] = all_meta["path"].apply(proc_path_str)
    return all_meta

In [ ]:
cf_str = """
infra:
  meta_col_order: ["mxa_code",  "m_num", "x_num", "a_num", "Barcode", "UM Accession", "Outside Accession", "Block", "Site/Organ", "Stain", "Diagnosis", "path", "tile", "comment", "qr_meta_fn"]
  out_fn: /nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa
mx_match:
  labportal: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List00000.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41178.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41179.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41180.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41181.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41182.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41183.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41184.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41185.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41186.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41187.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41188.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41189.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41190.csv"
    ]
  ocr: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241108v2.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241109v2.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241110v2.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241124mx.csv"
  ]
mxa_match:
  labportal: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41215.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41216.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41220.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41225.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41226.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41228.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41229.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41230.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41231.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41217.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41232.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41234.csv"
    #"/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41522.csv",
  ]
  ocr: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241121.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241122.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241123.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241124mxa.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241125mxa.csv"
  ]
a_match:
  labportal: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41522.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41524.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41525.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41547.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41700.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41648.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41649.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41650.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41674.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41675.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41713.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41714.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41740.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41743.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41744.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41774.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41775.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41778.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41792.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41793.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41794.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41800.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41801.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41802.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41823.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41825.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41847.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41850.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41897.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41923.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41925.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41928.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41933.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41935.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41936.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41938.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41955.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41957.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41963.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41982.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List41998.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42012.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42013.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42091.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42095.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42098.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42101.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42102.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42103.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42105.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42106.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42197.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42201.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42202.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42216.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42217.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42219.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42221.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42224.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42225.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42226.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42227.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42228.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42264.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42265.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42266.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42268.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42270.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42271.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42323.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42324.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42325.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42358.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42420.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42421.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42427.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42435.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42436.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42440.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42441.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42442.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42444.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42464.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42467.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42468.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42469.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42470.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42481.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42488.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42490.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42494.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42501.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42502.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42556.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42557.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42559.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42564.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42574.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42575.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42576.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42577.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42570.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42573.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42580.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42583.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42599.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42601.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42603.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42605.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42607.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42609.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42610.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42611.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42625.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42629.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42633.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42636.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42637.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42638.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42639.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42640.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42733.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42736.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42739.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42740.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42734.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42735.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42737.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42738.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42742.csv",    
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42743.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42873.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42874.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42876.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42879.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42875.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42877.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42878.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42928.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42931.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42933.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42934.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42988.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42991.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42992.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42993.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42997.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42998.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List42999.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43000.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43147.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43148.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43168.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43167.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43166.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43165.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43175.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43172.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43170.csv",  
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43169.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43178.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43179.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43180.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43181.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43182.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43183.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43184.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43185.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43188.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43572.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43575.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43581.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43582.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43584.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43585.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43586.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43587.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43588.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43589.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43591.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43594.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43595.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43676.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43678.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43679.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43680.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43681.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43682.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43683.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43684.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43685.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43705.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43710.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43720.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43724.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43888.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43889.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43909.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43915.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43916.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43913.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43917.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43918.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43921.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43922.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43923.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43924.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43925.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43926.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43927.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43928.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43929.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43958.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43959.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43960.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43976.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/mlins_scan_meta/List43978.csv",
    
  ]
  ocr: [
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241124a.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241125a.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241126.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241202.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241203.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241204.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241205.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/2024120607.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241208.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241209.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241210.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241211.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/2024121314.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241215.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241217.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241218.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241220.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241222.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241226.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241228.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241229.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241230.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20241231.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250101.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250103.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250104.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250105.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250106.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250107.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250108.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250109.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250110.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250111.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250112.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250113.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250114.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250115.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250117.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250118.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250119.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250121.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/2025012425.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250126.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250128.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250131.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250201.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250206.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250207.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250208.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250209.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250210.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250211.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250212.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250221.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250222.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250223.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250225.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250226.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250227.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250228.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250301.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250306.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250307.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250309.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250310.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250311.csv",
    "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/20250312.csv",
  ]
"""
 

In [ ]:
# We fill some values in the MxA code.
#   - -1 = this value does not exist, should not exist
#   - -2 = this value is missing. bad. (see problems below)

cf = yaml.safe_load(cf_str)
all_meta = process_metadata(cf)
print("OK")

In [ ]:
fsx_annotation = pd.read_csv("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/np_slide_ingest/annot_fsx/data/fsx_annot.csv")
mxa_fsx = set(fsx_annotation.loc[fsx_annotation["x_p"].isin({"x", "tp"}), "mxa"].tolist())
all_meta.loc[all_meta["mxa_code"].isin(mxa_fsx), "Block"] = all_meta.loc[all_meta["mxa_code"].isin(mxa_fsx), "Block"].str.removesuffix("x") + "x"
# all_meta[all_meta["Block"].str.endswith("x")]["Block"].value_counts()

In [ ]:
all_meta.drop("qr_meta_fn", axis=1).to_csv(f'{cf["infra"]["out_fn"]}.csv', index=False)
all_meta.drop("qr_meta_fn", axis=1).to_excel(f'{cf["infra"]["out_fn"]}.xlsx', index=False)

# Sanity checks - these tables should all be empty

## Known but not scanned - need to rescan

In [ ]:
pd.set_option('display.max_rows', None)  
print(len(all_meta[all_meta["path"].isna()]))
kns = all_meta[all_meta["path"].isna()]#.to_csv("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/known_not_scanned.csv", index=False)
kns if len(kns) else print("OK")


## Scanned but not known - need to upload / check LabPortal CSV

In [ ]:
snk = all_meta[all_meta["UM Accession"].isna()]#.to_csv("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/scanned_not_known.csv", index=False)
snk if len(snk) else print("OK")

In [ ]:
snkb = all_meta[all_meta["Block"].isna()]
snkb if len(snkb) else print("OK")

In [ ]:
check_select_stain = all_meta[all_meta["Stain"] == "Select Stain"]
check_select_stain if len(check_select_stain) else print("OK")

In [ ]:
#all_meta[all_meta["Block"]=="UNK"]

## Multiple scans by Mx numbers - need to decide which to keep
This includes the ones with A numbers. 
Just comment them out in the OCR results

In [ ]:
ms_mx = all_meta[(all_meta["m_num"]!=-1) & (all_meta["x_num"]!=-1)].groupby(["m_num", "x_num"]).filter(lambda x: len(x) > 1)
ms_mx if len(ms_mx) else print("OK")


## Multiple scans by A numbers only - need to decide which to keep

In [ ]:
ms_a = all_meta[(all_meta["m_num"]==-1) & (all_meta["x_num"]==-1)].groupby(["a_num"]).filter(lambda x: len(x) > 1)
ms_a if len(ms_a) else print("OK")


## M maps to multiple SU - need to give them a new M number
Choose one of the SU number and assign M{10000000 + orig_m_number} in labportal csv

In [ ]:
m_msu = all_meta.loc[all_meta["m_num"] !=-1 ,["m_num", "UM Accession"]].drop_duplicates().groupby("m_num").filter(lambda x: len(x) > 1)
m_msu if len(m_msu) else print("OK")

## SU maps to multiple M
Use the earlier M number, change the labportal csv and note the original Mx number as well

In [ ]:
su_mm = all_meta.loc[all_meta["m_num"] !=-1 ,["m_num", "UM Accession"]].drop_duplicates().groupby("UM Accession").filter(lambda x: len(x) > 1)
su_mm if len(su_mm) else print("OK")

## Make sure all files are there

In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"
for i,s in tqdm(all_meta.iterrows(), total = len(all_meta)):
    if type(s["path"]) == str:
        spath = opj(svs_root, s["path"])
        if not os.path.exists(spath):
            print(spath)
        assert os.path.exists(spath)
    else:
        print(f"DNE {s['mxa_code']}")

# Summary

In [ ]:
latest_qr_meta = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d.csv")

def pixel_alignment_filter(x):
    if "FS" in x["Block"]: return False
        
    block_stains = set(x["Stain"])
    if "H&E" not in block_stains: return False
    if not block_stains.difference({"H&E"}): return False
    return True

meta_stats = all_meta[~ (all_meta["path"].isna())]
is_frozen = meta_stats["Block"].fillna("").str.contains("FS")
not_frozen = ~ is_frozen # assume empty ones are frozen
is_latest = meta_stats["qr_meta_fn"] == latest_qr_meta

val_ct = pd.DataFrame(meta_stats.loc[:, "Stain"].value_counts())
val_ct.columns=["Count"]
total = val_ct["Count"].sum()
val_ct["Perct"] = val_ct["Count"].apply(lambda x: x / total * 100)


fz_ct = pd.DataFrame(meta_stats.loc[is_latest, "Stain"].value_counts())
fz_ct.columns=["Count"]
fz_total = fz_ct["Count"].sum()
fz_ct["Perct"] = fz_ct["Count"].apply(lambda x: x / fz_total * 100)

gby = meta_stats.groupby(["UM Accession", "Block"])["Stain"].agg(list).reset_index()
reg_cand = gby.loc[gby.apply(pixel_alignment_filter, axis=1)]

# Combine the top 20 rows with the "Others" row
top_20 = val_ct.reset_index().head(39)
others = val_ct.reset_index().iloc[39:].sum(numeric_only=True)
others['index'] = 'Other'
top20 = pd.concat([top_20, pd.DataFrame([others])], ignore_index=True)

if ((len(all_meta.loc[all_meta["UM Accession"]=="SU-15-65869"]) > 15) or 
    (len(all_meta.loc[all_meta["UM Accession"]=="SU-16-53059"]) > 15) or
    (len(all_meta.loc[all_meta["UM Accession"]=="SU-16-71632"]) > 21)):
    print("Check UNK block")

print(f'Total    Accessions | Blocks: {len(set(gby["UM Accession"]))} | {len(gby)}')
print(f'Reg cand Accessions | Blocks: {len(set(reg_cand["UM Accession"]))} | {len(reg_cand)}')
print(f"Frozen | permanent | total slides: {is_frozen.sum()} | {not_frozen.sum()} | {len(meta_stats)}\n")

print(f"Top stains:\t\t\t\tTop {latest_qr_meta} stains:")
for i, j in zip(str(val_ct[:25]).split("\n"), str(fz_ct[:25]).split("\n")):
    print(i+"\t\t"+j)

# get nio SU match meta
nio_su_match = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/code/chengjia/to_scan_sorted_nodup_inst.csv")
nio_su_match = nio_su_match[["Pathology Accession #", "tumor type "]].rename(columns={"tumor type ": "tumor", "Pathology Accession #": "su_num"}).drop_duplicates()

# get btb metadata
btb_to_pull = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/code/chengjia/btb_to_pull.csv")

# get phase 3 sus
phase3_to_pull = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/code/chengjia/scanning_phase3.csv")

# get all pulled and scanned SUs
pulled_sus = set(nio_su_match["su_num"]).union(set(btb_to_pull["Accession / ID #"]))
scanned_sus = set(all_meta["UM Accession"])
phase3_sus = set(phase3_to_pull["order"].tolist())

gpt_labels = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/code/chengjia/gpt_annot/gpt_annotation_v1v2.csv").drop(["brain_spine_nervous_origin", "tumor_diagnosis", "pathologist", "tumor_origin"], axis=1)
all_meta_with_tumor = all_meta.merge(gpt_labels,left_on="UM Accession", right_on="order", how="left").drop(["order"], axis=1)
all_meta_with_tumor["tumor_coarse_category"] = all_meta_with_tumor["tumor_coarse_category"].fillna("Unknown")
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Diffuse Gliomas", "tumor_coarse_category"] = "glioma"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Sellar Region Tumors", "tumor_coarse_category"] = "sellar"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Meningiomas", "tumor_coarse_category"] = "mening"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Metastatic tumors", "tumor_coarse_category"] = "met"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Circumscribed Astrocytic Gliomas", "tumor_coarse_category"] = "circm_astro"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Glioneuronal and Neuronal Tumors", "tumor_coarse_category"] = "glioneuronal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Non Neoplastic Brain Conditions", "tumor_coarse_category"] = "not_neo"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Embryonal Tumors", "tumor_coarse_category"] = "embryonal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Cranial and Paraspinal Nerve Tumors", "tumor_coarse_category"] = "nerve"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Unknown", "tumor_coarse_category"] = "unk"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Mesenchymal Non Meningothelial Tumors", "tumor_coarse_category"] = "mesenchymal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Ependymal Tumor", "tumor_coarse_category"] = "ependymal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Non CNS Specimens", "tumor_coarse_category"] = "not_cns"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Hematolymphoid Tumors", "tumor_coarse_category"] = "hematolymphoid"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Chondro-osseous Tumors", "tumor_coarse_category"] = "chondro"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Pineal Tumors", "tumor_coarse_category"] = "pineal"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Choroid Plexus Tumor", "tumor_coarse_category"] = "choroid"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Germ Cell Tumors", "tumor_coarse_category"] = "germ"
all_meta_with_tumor.loc[all_meta_with_tumor["tumor_coarse_category"]=="Melanocytic Tumors", "tumor_coarse_category"] = "melanocytic"

fig, [ax1, ax2, ax3] = plt.subplots(1,3,figsize=(20, 8))

v = venn3([scanned_sus, pulled_sus, phase3_sus], set_labels=["Scanned SUs", "Pulled SUs", "~TO PULL"],alpha=.5, ax=ax1)
v.get_patch_by_id("100").set_color("red")
v.get_patch_by_id("110").set_color("green")  
v.get_patch_by_id("010").set_color("orange")

slide_ct_by_class = all_meta_with_tumor[["UM Accession", "tumor_coarse_category"]].fillna("todo").groupby("tumor_coarse_category").count().reset_index()
labels = [f"{i} {j['UM Accession']}" for i, (_,j) in zip(slide_ct_by_class["tumor_coarse_category"], slide_ct_by_class.iterrows())]
wedges, texts = ax2.pie(slide_ct_by_class["UM Accession"], labels=labels, wedgeprops=dict(width=0.2), startangle=0)
ax2.set_title("Slide number class breakdown")

su_ct_by_class = all_meta_with_tumor[["UM Accession", "tumor_coarse_category"]].drop_duplicates().fillna("todo").groupby("tumor_coarse_category").count().reset_index()
labels = [f"{i} {j['UM Accession']}" for i, (_,j) in zip(su_ct_by_class["tumor_coarse_category"], su_ct_by_class.iterrows())]
wedges, texts = ax3.pie(su_ct_by_class["UM Accession"], labels=labels, wedgeprops=dict(width=0.2), startangle=0)
ax3.set_title("SU number class breakdown")

print("")
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("scanning_update.pdf")

In [ ]:
fig, ax1 = plt.subplots(1,1,figsize=(6, 6))

v = venn3([scanned_sus, pulled_sus, phase3_sus], set_labels=["Scanned SUs", "Pulled SUs", "~TO PULL"],alpha=.5, ax=ax1)
v.get_patch_by_id("100").set_color("red")
v.get_patch_by_id("110").set_color("green")  
v.get_patch_by_id("010").set_color("orange")
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("ns_venn.pdf")

In [ ]:
fig, [ax2, ax3] = plt.subplots(1,2,figsize=(12, 6))

slide_ct_by_class = all_meta_with_tumor[["UM Accession", "tumor_coarse_category"]].fillna("todo").groupby("tumor_coarse_category").count().reset_index()
labels = [f"{i} {j['UM Accession']}" for i, (_,j) in zip(slide_ct_by_class["tumor_coarse_category"], slide_ct_by_class.iterrows())]
wedges, texts = ax2.pie(slide_ct_by_class["UM Accession"], labels=labels, wedgeprops=dict(width=0.2), startangle=0)
ax2.set_title("Slide number class breakdown")

su_ct_by_class = all_meta_with_tumor[["UM Accession", "tumor_coarse_category"]].drop_duplicates().fillna("todo").groupby("tumor_coarse_category").count().reset_index()
labels = [f"{i} {j['UM Accession']}" for i, (_,j) in zip(su_ct_by_class["tumor_coarse_category"], su_ct_by_class.iterrows())]
wedges, texts = ax3.pie(su_ct_by_class["UM Accession"], labels=labels, wedgeprops=dict(width=0.2), startangle=0)
ax3.set_title("SU number class breakdown")

print("")
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("ns_class.pdf")

In [ ]:
stain_ct = val_ct.reset_index()
rows_to_sum = stain_ct["Count"]<50
summed_row = stain_ct[rows_to_sum].sum()
summed_row["Stain"] = "Others"
stain_ct = stain_ct[~rows_to_sum].reset_index(drop=True)
stain_ct.loc[len(stain_ct)] = summed_row

In [ ]:
stain_ct = val_ct.reset_index()
stain_ct["Stain"] = stain_ct['Stain'].str.cat(stain_ct['Count'].astype(str), sep=' ')
rows_to_sum = stain_ct["Count"]>=196
stain_ct.loc[~rows_to_sum, "Stain"] = ""

startangle = 90
labeldistance = 1.1
# If there are any small slices, determine which one is closest to the median value
if (~rows_to_sum).any():
    small_data = stain_ct.loc[~rows_to_sum]
    small_data_cumsum = small_data["Count"].cumsum()
    middle = small_data_cumsum.iloc[0] + (small_data_cumsum.iloc[-1]- small_data_cumsum.iloc[0]) / 2
    small_data.loc[small_data.iloc[(small_data_cumsum - middle).abs().argmin()].name, "Stain"] = f"Others {small_data['Count'].sum()}"
    stain_ct.loc[~rows_to_sum] = small_data
    
    
    data = stain_ct['Count'].values
    angles = 360 * data / data.sum()

    # Compute the starting angle for each slice using a cumulative sum
    start_angles = startangle + np.concatenate(([0], np.cumsum(angles)[:-1]))
    end_angles = startangle + np.cumsum(angles)
    
    small_indices = np.where(~rows_to_sum)[0]
    # Determine the arc's start angle from the first small slice and end angle from the last small slice
    first_idx = small_indices[0]
    last_idx = small_indices[-1]
    arc_start = start_angles[first_idx]
    arc_end = start_angles[last_idx] + angles[last_idx]  # end of the last wedge

In [ ]:
import matplotlib.patches as mpatches
import itertools

stain_perct  = stain_ct["Count"]/stain_ct["Count"].sum()



fig, ax4 = plt.subplots(1,1,figsize=(6, 6))
wedges, texts = ax4.pie(stain_ct["Count"], labels=stain_ct["Stain"], wedgeprops=dict(width=0.2), startangle=90, labeldistance=labeldistance)

if (~rows_to_sum).any():
    arc_radius = 1.03
    
    # Create an arc: width and height are 2*arc_radius because Arc takes a diameter
    arc = mpatches.Arc((0, 0),
                       width=2*arc_radius,
                       height=2*arc_radius,
                       angle=0,
                       theta1=arc_start,
                       theta2=arc_end,
                       color='lightgrey',
                       lw=4)

    ax4.add_patch(arc)
plt.tight_layout()
matplotlib.rcParams['pdf.fonttype'] = 42 
plt.savefig("ns_stain.pdf")


# Visualize image

In [ ]:
all_meta = pd.read_csv('/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv')

In [ ]:
currdf = all_meta.loc[all_meta["UM Accession"]=="SU-20-28530"]
currdf = currdf.loc[currdf["Block"]=="C6"]
#currdf = currdf.loc[currdf["Block"].str.contains("FS")]
#currdf = currdf.loc[currdf["Stain"]=="H&E"]
#currdf = all_meta.loc[all_meta["a_num"].isin(range(11917,11925))]
#currdf = all_meta.loc[all_meta["Barcode"]=="S1230CLXM"]
#currdf = all_meta[(all_meta["m_num"]==-1) & (all_meta["x_num"]==-1)].groupby(["a_num"]).filter(lambda x: len(x) > 1)
#currdf = all_meta.loc[all_meta["a_num"]==21358]
#currdf = all_meta.loc[all_meta["a_num"].isin({18540,18541,18632,18634,18635,18659})]

#currdf = all_meta[all_meta["qr_meta_fn"]=="20250209.csv"]
#currdf = currdf.sort_values(["Block", "Stain"])
currdf


In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"

for i, s in tqdm(currdf.fillna("").iterrows(), total=len(currdf)):
    if s["path"] == "":
        continue
        
    fig, axs = plt.subplots(1,3,  figsize=(12,3), width_ratios=[1, 3,3])
    
    try:
        slide = openslide.OpenSlide(opj(svs_root, s["path"]))

        axs[0].imshow(slide.associated_images["label"])
        axs[0].set_title(s["mxa_code"])
        axs[1].imshow(slide.associated_images["macro"])
        axs[2].imshow(slide.associated_images["thumbnail"])
        axs[1].set_title(s["path"].split("/")[-1]+"\n"+s['UM Accession']+" / "+s['Block']+" / "+s["Stain"])
    except:
        axs[0].set_title(s["mxa_code"])
        axs[1].set_title(s["path"].split("/")[-1]+"\n"+s['UM Accession']+" / "+s['Block']+" / "+s["Stain"])
        
    for ax in axs: ax.axis("off") 
    
    subs_str = "_".join([s['UM Accession'], s['Block'], sanitize_string(s["Stain"])]).lower()
    
    #slide.associated_images["macro"].save(f"{subs_str}_macro.png")
    #slide.associated_images["thumbnail"].save(f"{subs_str}_thumbnail.png")    